## paso 1 Recolección de datos (conexión a MariaDB)



In [ ]:
# Si falta algun paquete en tu entorno local: pip install pymysql pandas numpy scikit-learn matplotlib

import json
import statistics
from collections import defaultdict
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pymysql
from sklearn.cluster import DBSCAN

conexion base de datos

In [ ]:
def load_env(path):
    """Parser minimo de .env (KEY=VALUE), sin dependencias extra."""
    values = {}
    for line in Path(path).read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        values[key.strip()] = value.strip().strip('"').strip("'")
    return values


def find_env(start=None, filename=".env"):
    """Busca .env subiendo carpetas desde el cwd del kernel, sin asumir una profundidad fija."""
    path = Path(start or Path.cwd()).resolve()
    for candidate in [path, *path.parents]:
        if (candidate / filename).exists():
            return candidate / filename
    raise FileNotFoundError(f"No se encontro {filename} subiendo desde {path}")


def conectar():
    """Abre una conexion nueva a MariaDB con las credenciales del .env. Usar con 'with'."""
    return pymysql.connect(
        host="127.0.0.1", port=3306,
        user=env["MARIADB_USER"], password=env["MARIADB_PASSWORD"], database=env["MARIADB_DATABASE"],
        charset="utf8mb4", cursorclass=pymysql.cursors.DictCursor,
    )


env = load_env(find_env())  # credenciales de MariaDB

with conectar() as connection:
    print("Conexion OK a la base:", env["MARIADB_DATABASE"])


In [ ]:
with conectar() as connection, connection.cursor() as cursor:
    cursor.execute("SELECT * FROM raw_gps LIMIT 20")
    filas = cursor.fetchall()

tabla = pd.DataFrame(filas)
tabla

In [ ]:
# --- Rango de fechas editable (maximo 14 dias, ambos dias incluidos) ---
FECHA_DESDE = "2026-04-01"  # YYYY-MM-DD
FECHA_HASTA = "2026-04-14"  # YYYY-MM-DD
MAX_DIAS = 14

desde, hasta = pd.Timestamp(FECHA_DESDE), pd.Timestamp(FECHA_HASTA)
dias = (hasta - desde).days + 1
if dias < 1:
    raise ValueError("FECHA_HASTA tiene que ser igual o posterior a FECHA_DESDE")
if dias > MAX_DIAS:
    raise ValueError(f"El rango es de {dias} dias; el maximo es {MAX_DIAS}")

# Todas las geolocalizaciones de cada patente en el rango. DISTINCT quita las filas que la
# ETL repite (mismo reporte copiado) sin perder ningun punto real.
with conectar() as connection, connection.cursor() as cursor:
    cursor.execute(
        """
        SELECT DISTINCT asset_id, asset_name, report_date, latitude, longitude, speed, event_name
        FROM raw_gps
        WHERE report_date >= %s AND report_date < %s
          AND latitude IS NOT NULL AND longitude IS NOT NULL
        ORDER BY asset_id, report_date
        """,
        (desde, hasta + pd.Timedelta(days=1)),
    )
    filas = cursor.fetchall()

df = pd.DataFrame(filas)
df["report_date"] = pd.to_datetime(df["report_date"])
print(f"{len(df):,} puntos de {df['asset_id'].nunique()} patentes entre {FECHA_DESDE} y {FECHA_HASTA} ({dias} dias)")

# Puntos por patente
resumen_patentes = (df.groupby("asset_id")
                    .agg(nombre=("asset_name", "first"), puntos=("report_date", "size"),
                         primer_punto=("report_date", "min"), ultimo_punto=("report_date", "max"))
                    .sort_values("puntos", ascending=False))
display(resumen_patentes)

**herramientas de mapas**

In [ ]:
# Herramientas que usan los 3 mapas de abajo (calor, rutas y uso de rutas).

import folium
from folium.plugins import HeatMap
from branca.element import MacroElement
from jinja2 import Template


# ---------------------------------------------------------------------------
# Respaldo automatico: si el mapa base activo falla, cambia al siguiente proveedor
# ---------------------------------------------------------------------------
class RespaldoMapaBase(MacroElement):
    _template = Template("""
    {% macro script(this, kwargs) %}
    (function () {
      var mapa = {{ this._parent.get_name() }};
      var capas = [{% for c in this.capas %}{{ c.get_name() }}{% if not loop.last %}, {% endif %}{% endfor %}];
      var cambios = 0;
      capas.forEach(function (capa, i) {
        var errores = 0, cargas = 0;
        capa.on('tileload', function () { cargas++; });
        capa.on('tileerror', function () {
          errores++;
          if (errores >= {{ this.umbral }} && errores > cargas && mapa.hasLayer(capa)
              && cambios < capas.length - 1) {
            cambios++;
            mapa.removeLayer(capa);
            mapa.addLayer(capas[(i + 1) % capas.length]);
            console.warn('Mapa base con fallas, cambiando de proveedor');
          }
        });
        capa.on('add', function () { errores = 0; cargas = 0; });
      });
    })();
    {% endmacro %}
    """)

    def __init__(self, capas, umbral=6):
        super().__init__()
        self._name = "RespaldoMapaBase"
        self.capas = capas
        self.umbral = umbral


# ---------------------------------------------------------------------------
# Mapas base
# ---------------------------------------------------------------------------
MAPAS_BASE = {
    "Claro (Esri)": dict(
        tiles="https://server.arcgisonline.com/ArcGIS/rest/services/Canvas/World_Light_Gray_Base/MapServer/tile/{z}/{y}/{x}",
        attr="Tiles &copy; Esri &mdash; Esri, HERE, Garmin, OpenStreetMap contributors",
        max_native_zoom=16,  # este mapa llega hasta zoom 16; sobre eso amplia el ultimo nivel
    ),
    # Respaldo con el mismo estilo claro, por si Esri no responde
    "Claro (CARTO)": dict(
        tiles="https://{s}.basemaps.cartocdn.com/light_all/{z}/{x}/{y}{r}.png",
        attr="&copy; OpenStreetMap contributors &copy; CARTO",
        subdomains="abcd",
        max_native_zoom=19,
    ),
    "Satelite (Esri)": dict(
        tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
        attr="Tiles &copy; Esri &mdash; Esri, Maxar, Earthstar Geographics, GIS User Community",
        max_native_zoom=18,  # sobre 18 amplia el ultimo nivel en vez de mostrar tiles vacios
    ),
}


def nuevo_mapa(*mapas_base, attr=None):
    """Mapa vacio con los mapas base indicados (solo el primero queda visible).
    attr reemplaza el credito de los mapas base; None = el credito completo de cada uno."""
    mapa = folium.Map(tiles=None, prefer_canvas=True, control_scale=True, max_zoom=19)
    capas = []
    for i, nombre in enumerate(mapas_base):
        opciones = dict(MAPAS_BASE[nombre], name=nombre, max_zoom=19, show=i == 0)
        if attr is not None:
            opciones["attr"] = attr
        capas.append(folium.TileLayer(**opciones).add_to(mapa))
    return mapa, capas


def gps_validos(g):
    """Quita coordenadas fuera de rango y el 0,0 (GPS sin fix)."""
    return g[g["latitude"].between(-90, 90) & g["longitude"].between(-180, 180)
             & ~((g["latitude"] == 0) & (g["longitude"] == 0))]


def agregar_faenas(mapa, faenas):
    """Capa 'Faenas' con el contorno de cada faena y su nombre al pasar el mouse."""
    capa = folium.FeatureGroup(name="Faenas").add_to(mapa)
    for faena in faenas:
        folium.GeoJson(
            faena["geojson"],
            style_function=lambda _: {"color": "green", "weight": 2, "fillOpacity": 0.15},
            tooltip=faena["nombre"],
        ).add_to(capa)
    return capa

**faenas**

In [ ]:
# ---------------------------------------------------------------------------
# Faenas: MySQL entrega GeoJSON
# ---------------------------------------------------------------------------
with conectar() as faenas_connection, faenas_connection.cursor() as faenas_cursor:
    faenas_cursor.execute("""
        SELECT id, nombre, area_m2, ST_AsGeoJSON(geocerca, 6) AS geojson
        FROM faenas
        WHERE geocerca IS NOT NULL
        ORDER BY id
    """)
    faenas = faenas_cursor.fetchall()


def centro_geojson(geom):
    """Centroide (lat, lon) del anillo exterior mas grande; soporta Polygon y MultiPolygon."""
    poligonos = [geom["coordinates"]] if geom["type"] == "Polygon" else geom["coordinates"]
    mejor, mejor_area = None, -1.0
    for poligono in poligonos:
        anillo = poligono[0]  # anillo exterior, puntos [lon, lat]
        a = cx = cy = 0.0
        for (x0, y0), (x1, y1) in zip(anillo, anillo[1:] + anillo[:1]):
            cruz = x0 * y1 - x1 * y0
            a += cruz
            cx += (x0 + x1) * cruz
            cy += (y0 + y1) * cruz
        if abs(a) < 1e-15:
            lon = sum(p[0] for p in anillo) / len(anillo)
            lat = sum(p[1] for p in anillo) / len(anillo)
        else:
            lon, lat = cx / (3 * a), cy / (3 * a)
        if abs(a) > mejor_area:
            mejor, mejor_area = (lat, lon), abs(a)
    return mejor


for faena in faenas:
    g = faena["geojson"]
    faena["geojson"] = g if isinstance(g, dict) else json.loads(g)
    faena["lat"], faena["lon"] = centro_geojson(faena["geojson"])

**mapa de calor**

In [ ]:
# Mapa de calor con TODOS los reportes de todas las patentes: cada reporte suma en su
# celda de ~100 m (peso logaritmico). No se descarta ninguna celda.
coords = gps_validos(df[["latitude", "longitude"]].dropna())
celdas = coords.round(3).value_counts().rename("n").reset_index()
celdas["peso"] = np.log1p(celdas["n"]) / np.log1p(celdas["n"].max())

# Orden de respaldo: Claro (Esri) -> Claro (CARTO) -> Satelite
mapa, capas_base = nuevo_mapa("Claro (Esri)", "Claro (CARTO)", "Satelite (Esri)")

print(f"Mapa de calor: {len(coords):,} reportes de {df['asset_id'].nunique()} patentes en {len(celdas):,} celdas")
HeatMap(celdas[["latitude", "longitude", "peso"]].values.tolist(),
        name="Densidad GPS", radius=8, blur=10, min_opacity=0.3).add_to(mapa)

capa_faenas = folium.FeatureGroup(name="Faenas").add_to(mapa)
for faena in faenas:
    area = f"{faena['area_m2']:.0f} m2" if faena["area_m2"] is not None else "s/d"
    etiqueta = f"<b>{faena['nombre']}</b><br>area: {area}"
    folium.GeoJson(
        faena["geojson"],
        style_function=lambda _: {"color": "green", "weight": 2, "fillOpacity": 0.2},
        popup=folium.Popup(etiqueta, max_width=250),
    ).add_to(capa_faenas)
    folium.Marker(
        [faena["lat"], faena["lon"]],
        tooltip=faena["nombre"],
        popup=folium.Popup(etiqueta, max_width=250),
        icon=folium.Icon(color="green", icon="industry", prefix="fa"),
    ).add_to(capa_faenas)

if faenas:
    lats = [f["lat"] for f in faenas]
    lons = [f["lon"] for f in faenas]
    mapa.fit_bounds([[min(lats), min(lons)], [max(lats), max(lons)]], padding=(40, 40), max_zoom=12)
else:
    mapa.location, mapa.zoom_start = [-33.45, -70.66], 6

RespaldoMapaBase(capas_base).add_to(mapa)
folium.LayerControl(collapsed=False).add_to(mapa)

mapa

In [ ]:
# Rutas por camion con TODOS los reportes de todas las patentes del rango de fechas de arriba.

# --- Parametros editables ---
ASSET_IDS = None        # None = todas las patentes; o una lista, ej. ["RYCV17", "SWDL56"]
GAP_MINUTOS = 30        # pausa mayor a esto corta la ruta en un nuevo viaje

gps = gps_validos(df[["asset_id", "report_date", "latitude", "longitude"]].dropna())
if ASSET_IDS is not None:
    gps = gps[gps["asset_id"].isin(ASSET_IDS)]
gps = gps.sort_values(["asset_id", "report_date"])
camiones = gps["asset_id"].value_counts().index

mapa_rutas, _ = nuevo_mapa("Claro (Esri)", "Satelite (Esri)", attr="Tiles &copy; Esri")

PALETA = ["#e6194b", "#3cb44b", "#4363d8", "#f58231", "#911eb4", "#008080", "#9a6324",
          "#800000", "#808000", "#000075", "#f032e6", "#469990", "#c49a00", "#005f73", "#ae2012"]

puntos_dibujados = 0
for k, asset in enumerate(camiones):
    color = PALETA[k % len(PALETA)]
    t = gps[gps["asset_id"] == asset]
    grupo = folium.FeatureGroup(name=f"Camion {asset}").add_to(mapa_rutas)
    viaje = (t["report_date"].diff() > pd.Timedelta(minutes=GAP_MINUTOS)).cumsum()
    # Todos los viajes del camion van en una sola linea (multi-linea): el mapa pesa mucho
    # menos. 5 decimales = ~1 m.
    tramos, sueltos = [], []
    for _, tramo in t.groupby(viaje):
        pts = [(round(lat, 5), round(lon, 5)) for lat, lon in tramo[["latitude", "longitude"]].itertuples(index=False)]
        puntos_dibujados += len(pts)
        (tramos if len(pts) >= 2 else sueltos).append(pts)
    info = f"Camion {asset} · {viaje.nunique()} viajes<br>{t['report_date'].min():%d-%m %H:%M} → {t['report_date'].max():%d-%m %H:%M}"
    if tramos:
        folium.PolyLine(tramos, color=color, weight=3, opacity=0.8, tooltip=info).add_to(grupo)
    for pts in sueltos:  # viaje de un solo reporte: sin circulo no se veria en el mapa
        folium.CircleMarker(pts[0], radius=3, color=color, fill=True, fill_opacity=1,
                            tooltip=info).add_to(grupo)

agregar_faenas(mapa_rutas, faenas)

mapa_rutas.fit_bounds([[gps["latitude"].min(), gps["longitude"].min()],
                       [gps["latitude"].max(), gps["longitude"].max()]], padding=(30, 30))
folium.LayerControl(collapsed=True).add_to(mapa_rutas)  # 112 camiones: la lista se abre con el icono de capas

print(f"Rutas: {puntos_dibujados:,} de {len(gps):,} reportes · {len(camiones)} patentes")
mapa_rutas

In [ ]:
# Mapa de calor por linea con TODOS los reportes de 'df' (rango FECHA_DESDE..FECHA_HASTA de arriba).
# Si cambias las fechas arriba y vuelves a ejecutar, este mapa se actualiza solo.
#
# Cada linea tiene un "radio invisible" alrededor: cuenta cuantos camiones distintos
# van por su lado (paralelos, en direccion parecida), no solo los que pasan justo encima.
# El radio crece con el largo de la linea: linea larga = GPS impreciso = radio mayor.

# --- Parametros de union de reportes ---
MAX_DIST_M = None     # None = se unen todos los reportes seguidos; o un tope en metros, ej. 300
MAX_GAP_MIN = 5       # si el camion dejo de reportar mas de estos minutos, no se une (no se inventa recorrido)
VEL_MAX_KMH = 150     # saltos de GPS mas rapidos que esto no se unen (no son recorridos reales)

# --- Parametros del radio invisible ---
RADIO_BASE_M = 60     # radio minimo alrededor de cada linea (donde el GPS es preciso)
RADIO_FACTOR = 0.15   # el radio crece con el largo: linea de 2 km -> radio 360 m
RADIO_MAX_M = 800     # tope del radio
PUNTOS_LINEA = 9      # puntos de control a lo largo de cada linea
FRACCION_MIN = 0.6    # otro camion cuenta si esta cerca en al menos el 60% de la linea...
ANG_MAX = 30          # ...y va en direccion parecida (grados), asi los cruces no suman
PASO_M = 50           # resolucion de la nube de referencia (m)
N_RADIOS = 8          # el radio de cada linea se redondea a uno de estos tamanos (mas = mas exacto y lento)
CELDAS_RADIO = 3      # el radio se cubre con esta cantidad de celdas (mas = mas exacto y lento)
ANG_PASO = 15         # las direcciones se agrupan en bloques de estos grados
LOTE = 5000           # lineas por lote (bajalo si se queda sin memoria)

# Reportes del rango de fechas, sin coordenadas invalidas, ordenados por camion y hora
g = gps_validos(df[["asset_id", "report_date", "latitude", "longitude"]].dropna())
n_filas = len(g)
g = g.drop_duplicates(["asset_id", "report_date"])  # la ETL repite el mismo reporte (misma hora y posicion)
g = g.sort_values(["asset_id", "report_date"])

# Cada reporte con el anterior del mismo camion
prev = g.groupby("asset_id")[["latitude", "longitude", "report_date"]].shift()
dt_h = (g["report_date"] - prev["report_date"]).dt.total_seconds() / 3600
dist_m = np.hypot(g["latitude"] - prev["latitude"],
                  (g["longitude"] - prev["longitude"]) * np.cos(np.radians(g["latitude"]))) * 111_000
hay_prev = prev["latitude"].notna()

# Se une cada reporte con el anterior del mismo camion, salvo pausas largas sin reportar
# o saltos de GPS imposibles (mas rapidos que VEL_MAX_KMH)
unir = (hay_prev & (dt_h > 0) & (dt_h <= MAX_GAP_MIN / 60) & (dist_m > 0)
        & (dist_m / 1000 <= VEL_MAX_KMH * dt_h))
if MAX_DIST_M is not None:
    unir &= dist_m <= MAX_DIST_M

lat0, lon0 = prev.loc[unir, "latitude"].to_numpy(), prev.loc[unir, "longitude"].to_numpy()
lat1, lon1 = g.loc[unir, "latitude"].to_numpy(), g.loc[unir, "longitude"].to_numpy()
camion = pd.factorize(g.loc[unir, "asset_id"])[0]

# --- Radio invisible por linea ---
# Pasar a metros (plano local) para medir distancias y angulos reales
lat_ref = np.radians(np.mean(np.r_[lat0, lat1]))
kx, ky = 111_000 * np.cos(lat_ref), 111_000
x0, y0, x1, y1 = lon0 * kx, lat0 * ky, lon1 * kx, lat1 * ky
dx, dy = x1 - x0, y1 - y0
largo = np.hypot(dx, dy)
ang = np.degrees(np.arctan2(dy, dx)) % 180          # direccion sin sentido (ida = vuelta)
radio = np.minimum(RADIO_BASE_M + RADIO_FACTOR * largo, RADIO_MAX_M)

def unir_mascaras(claves, mascaras):
    """Junta (OR) las mascaras de camiones que caen en la misma clave."""
    orden = np.argsort(claves, kind="stable")
    claves, mascaras = claves[orden], mascaras[orden]
    ini = np.flatnonzero(np.r_[True, claves[1:] != claves[:-1]])
    return claves[ini], np.bitwise_or.reduceat(mascaras, ini, axis=0)

# Nube de referencia: cada linea muestreada cada PASO_M, con su camion y direccion
pasos = np.maximum(1, np.ceil(largo / PASO_M)).astype(int)
n_pts = pasos + 1
par = np.repeat(np.arange(len(pasos)), n_pts)
t = (np.arange(n_pts.sum()) - np.repeat(np.cumsum(n_pts) - n_pts, n_pts)) / pasos[par]
nube_x, nube_y = x0[par] + dx[par] * t, y0[par] + dy[par] * t
N_ANG = 180 // ANG_PASO
ang_bin = (ang // ANG_PASO).astype(np.int64) % N_ANG
nube_ang, nube_cam = ang_bin[par], camion[par]

# Cada camion es un bit: 112 camiones caben en 2 enteros de 64 bits por celda
n_cam = camion.max() + 1
W = (n_cam + 63) // 64
bit_camion = np.zeros((n_cam, W), dtype=np.uint64)
bit_camion[np.arange(n_cam), np.arange(n_cam) // 64] = np.left_shift(np.uint64(1), (np.arange(n_cam) % 64).astype(np.uint64))

# Direcciones que se consideran "parecidas" (desplazamientos de bin de angulo)
d_ang = np.arange(-(N_ANG // 2), N_ANG // 2 + 1)
d_ang = d_ang[np.abs(d_ang) * ANG_PASO <= ANG_MAX]
# Celdas vecinas dentro del circulo de radio CELDAS_RADIO
di, dj = np.mgrid[-CELDAS_RADIO:CELDAS_RADIO + 1, -CELDAS_RADIO:CELDAS_RADIO + 1]
disco = np.c_[di.ravel(), dj.ravel()][(di ** 2 + dj ** 2).ravel() <= CELDAS_RADIO ** 2]

# El radio de cada linea se redondea a uno de N_RADIOS tamanos; cada tamano usa una
# grilla de celda = radio / CELDAS_RADIO, asi el vecindario siempre es de pocas celdas
radios = np.geomspace(RADIO_BASE_M, RADIO_MAX_M, N_RADIOS)
nivel = np.abs(np.log(radio)[:, None] - np.log(radios)[None, :]).argmin(1)

tk = (np.arange(PUNTOS_LINEA) + 0.5) / PUNTOS_LINEA
min_puntos = int(np.ceil(FRACCION_MIN * PUNTOS_LINEA))
camiones_cerca = np.ones(len(largo), dtype=int)
x_min, y_min = min(x0.min(), x1.min()), min(y0.min(), y1.min())

for k, r in enumerate(radios):
    lineas_k = np.flatnonzero(nivel == k)
    if len(lineas_k) == 0:
        continue
    celda = r / CELDAS_RADIO
    ny = int((max(y0.max(), y1.max()) - y_min) / celda) + 2 * CELDAS_RADIO + 3
    clave = lambda gx, gy, a: ((gx + CELDAS_RADIO + 1) * ny + (gy + CELDAS_RADIO + 1)) * N_ANG + a

    # Tabla celda+direccion -> camiones que pasan por ahi (con direcciones parecidas ya sumadas)
    gx = ((nube_x - x_min) // celda).astype(np.int64)
    gy = ((nube_y - y_min) // celda).astype(np.int64)
    c_tab, m_tab = unir_mascaras(clave(gx, gy, nube_ang), bit_camion[nube_cam])
    a_tab = c_tab % N_ANG
    base = c_tab - a_tab
    c_tab, m_tab = unir_mascaras(np.concatenate([base + (a_tab + d) % N_ANG for d in d_ang]),
                                 np.tile(m_tab, (len(d_ang), 1)))

    # Puntos de control de cada linea: camiones en las celdas vecinas (radio de la linea)
    for ini in range(0, len(lineas_k), LOTE):
        s = lineas_k[ini:ini + LOTE]
        qx = ((x0[s, None] + dx[s, None] * tk - x_min) // celda).astype(np.int64)
        qy = ((y0[s, None] + dy[s, None] * tk - y_min) // celda).astype(np.int64)
        qa = np.broadcast_to(ang_bin[s, None], qx.shape)
        mascara = np.zeros(qx.shape + (W,), dtype=np.uint64)
        for ddx, ddy in disco:
            q = clave(qx + ddx, qy + ddy, qa)
            pos = np.minimum(np.searchsorted(c_tab, q), len(c_tab) - 1)
            hallado = c_tab[pos] == q
            mascara[hallado] |= m_tab[pos[hallado]]
        # Por camion: en cuantos puntos de control esta cerca
        bits = np.unpackbits(mascara.view(np.uint8), axis=-1)       # (lineas, puntos, bits)
        cobertura = bits.sum(axis=1)
        camiones_cerca[s] = np.maximum((cobertura >= min_puntos).sum(axis=1), 1)

lineas_uso = pd.DataFrame({
    "lat0": lat0.round(5), "lon0": lon0.round(5), "lat1": lat1.round(5), "lon1": lon1.round(5),
    "camiones": camiones_cerca,
}).drop_duplicates()

# --- Resumen ---
pares = dist_m[hay_prev & (dist_m > 0)]
print(f"{FECHA_DESDE} a {FECHA_HASTA}: {n_filas:,} filas -> {len(g):,} reportes reales "
      f"({n_filas - len(g):,} copias eliminadas) de {g['asset_id'].nunique()} camiones")
print(f"Distancia entre reportes seguidos: mediana {pares.median():.0f} m · 90% bajo {pares.quantile(0.9):.0f} m")
print(f"Lineas dibujadas: {unir.sum():,} · no unidas (pausa larga o salto de GPS): {len(pares) - unir.sum():,}")
print(f"Radio de las lineas: mediana {np.median(radio):.0f} m · maximo {radio.max():.0f} m")

# --- Mapa ---
mapa_uso, _ = nuevo_mapa("Claro (Esri)", attr="Tiles &copy; Esri")

# Azul = poco concurrido ... rojo = muy concurrido
max_camiones = lineas_uso["camiones"].max()
escala = folium.LinearColormap(["#2c7bb6", "#00a6ca", "#90eb9d", "#f9d057", "#f29e2e", "#d7191c"],
                               vmin=1, vmax=max(max_camiones, 2),
                               caption="Camiones distintos que van por la zona (azul = pocos, rojo = muchos)")
escala.add_to(mapa_uso)

# Las lineas con la misma concentracion van en una sola multi-linea: el navegador no se
# cuelga aunque sean decenas de miles. Se dibuja de menos a mas para que lo rojo quede encima.
capa_uso = folium.FeatureGroup(name="Uso de rutas").add_to(mapa_uso)
for n_camiones, grupo in sorted(lineas_uso.groupby("camiones"), key=lambda x: x[0]):
    folium.PolyLine(
        [[(r.lat0, r.lon0), (r.lat1, r.lon1)] for r in grupo.itertuples(index=False)],
        color=escala(n_camiones), weight=2 + 4 * n_camiones / max_camiones, opacity=0.9,
        tooltip=f"{n_camiones} camiones distintos van por aqui",
    ).add_to(capa_uso)

agregar_faenas(mapa_uso, faenas)

mapa_uso.fit_bounds([[lineas_uso[["lat0", "lat1"]].min().min(), lineas_uso[["lon0", "lon1"]].min().min()],
                     [lineas_uso[["lat0", "lat1"]].max().max(), lineas_uso[["lon0", "lon1"]].max().max()]],
                    padding=(30, 30))
folium.LayerControl(collapsed=False).add_to(mapa_uso)

mapa_uso

In [ ]:
# Mapa de puntos: cada reporte GPS de 'df' como un punto, sin unir lineas y sin descartar reportes.
# Color = cuantos reportes hay alrededor (azul = pocos, rojo = muchos), como el mapa de uso de rutas.
# Al pasar el mouse sobre un punto muestra que camiones reportaron ahi (dia y hora);
# al sacar el mouse el cartel desaparece.
# Usa el mismo rango FECHA_DESDE..FECHA_HASTA de arriba.
#
# Los puntos se mandan al navegador como numeros compactos y se dibujan todos en un solo
# canvas: con cientos de miles de reportes, un marcador por punto (GeoJSON) cuelga el navegador.

# --- Parametros editables ---
RADIO_PUNTO = 4             # tamano del punto en pixeles
CELDA_M = 50                # resolucion para medir la concentracion de puntos (m)
RADIO_CELDAS = 1            # se suman los puntos de las celdas vecinas hasta este radio
MAX_CAMIONES_CARTEL = 10    # camiones listados en el cartel; el resto se resume como "y N mas"

DIAS = ["Lunes", "Martes", "Miércoles", "Jueves", "Viernes", "Sábado", "Domingo"]
# Azul = pocos reportes alrededor ... rojo = muchos (mismos colores que el mapa de uso de rutas)
COLORES = ["#2c7bb6", "#00a6ca", "#90eb9d", "#f9d057", "#f29e2e", "#d7191c"]


class CapaPuntos(MacroElement):
    """Dibuja todos los puntos en un canvas y, al pasar el mouse, muestra que camiones reportaron ahi."""
    _template = Template("""
    {% macro script(this, kwargs) %}
    (function () {
      var mapa = {{ this._parent.get_name() }};
      var d = {{ this.datos }};
      var n = d.lat.length, radio = {{ this.radio }}, maxCamiones = {{ this.max_camiones }};
      // Coordenadas del mundo a zoom 0 (se calculan una vez; a otro zoom solo se escalan)
      var wx = new Float64Array(n), wy = new Float64Array(n);
      for (var i = 0; i < n; i++) {
        var p = mapa.project([d.lat[i] / 1e5, d.lon[i] / 1e5], 0);
        wx[i] = p.x; wy[i] = p.y;
      }
      var canvas = L.DomUtil.create('canvas', 'leaflet-zoom-hide');
      canvas.style.pointerEvents = 'none';
      mapa.getPanes().overlayPane.appendChild(canvas);
      // Celda de pantalla -> TODOS los puntos que caen en ella (tambien los encimados)
      var celdas = new Map(), CELDA = 8, px = new Float32Array(n), py = new Float32Array(n);

      function dibujar() {
        var tam = mapa.getSize(), esc = Math.pow(2, mapa.getZoom()), orig = mapa.getPixelBounds().min;
        canvas.width = tam.x; canvas.height = tam.y;
        L.DomUtil.setPosition(canvas, mapa.containerPointToLayerPoint([0, 0]));
        var ctx = canvas.getContext('2d');
        ctx.globalAlpha = 0.8;
        celdas.clear();
        var caminos = d.colores.map(function () { return new Path2D(); });
        for (var i = 0; i < n; i++) {
          var x = wx[i] * esc - orig.x, y = wy[i] * esc - orig.y;
          if (x < -radio || y < -radio || x > tam.x + radio || y > tam.y + radio) continue;
          px[i] = x; py[i] = y;
          caminos[d.niv[i]].moveTo(x + radio, y);
          caminos[d.niv[i]].arc(x, y, radio, 0, 2 * Math.PI);
          var k = Math.floor(x / CELDA) * 100000 + Math.floor(y / CELDA), lista = celdas.get(k);
          if (lista) lista.push(i); else celdas.set(k, [i]);
        }
        // De menos a mas concentrado, para que lo rojo quede encima
        caminos.forEach(function (c, k) { ctx.fillStyle = d.colores[k]; ctx.fill(c); });
      }

      function dosCifras(v) { return (v < 10 ? '0' : '') + v; }
      function cuando(i) {
        var s = d.seg[i];
        return d.dias[d.dia[i]] + ' ' + dosCifras(Math.floor(s / 3600)) + ':'
          + dosCifras(Math.floor(s / 60) % 60) + ':' + dosCifras(s % 60);
      }
      function momento(i) { return d.dia[i] * 86400 + d.seg[i]; }

      // Un solo cartel que sigue al mouse y se quita apenas el mouse sale de los puntos
      var cartel = L.tooltip({direction: 'top', offset: [0, -radio - 2], opacity: 0.95});
      function cerrar() { if (mapa.hasLayer(cartel)) mapa.removeLayer(cartel); }

      mapa.on('mousemove', function (e) {
        var cx = Math.floor(e.containerPoint.x / CELDA), cy = Math.floor(e.containerPoint.y / CELDA);
        var alcance = (radio + 2) * (radio + 2), porCamion = new Map(), total = 0;
        for (var a = -1; a <= 1; a++) for (var b = -1; b <= 1; b++) {
          var lista = celdas.get((cx + a) * 100000 + cy + b);
          if (!lista) continue;
          for (var j = 0; j < lista.length; j++) {
            var i = lista[j], ddx = px[i] - e.containerPoint.x, ddy = py[i] - e.containerPoint.y;
            if (ddx * ddx + ddy * ddy > alcance) continue;
            var r = porCamion.get(d.cam[i]);
            if (!r) porCamion.set(d.cam[i], r = {n: 0, primero: i, ultimo: i});
            r.n++; total++;
            if (momento(i) < momento(r.primero)) r.primero = i;
            if (momento(i) > momento(r.ultimo)) r.ultimo = i;
          }
        }
        if (total === 0) { cerrar(); return; }
        // Todos los camiones bajo el mouse, el que mas reporto ahi primero
        var filas = Array.from(porCamion.entries()).sort(function (x, y) { return y[1].n - x[1].n; });
        var html = '<b>' + total + (total === 1 ? ' reporte' : ' reportes') + ' de ' + filas.length
          + (filas.length === 1 ? ' camión' : ' camiones') + '</b>';
        filas.slice(0, maxCamiones).forEach(function (f) {
          var r = f[1];
          html += '<br><b>' + d.camiones[f[0]] + ':</b> ' + (r.n === 1 ? cuando(r.primero)
            : r.n + ' reportes, ' + cuando(r.primero) + ' → ' + cuando(r.ultimo));
        });
        if (filas.length > maxCamiones) html += '<br>… y ' + (filas.length - maxCamiones) + ' camiones más';
        cartel.setLatLng(e.latlng).setContent(html);
        if (!mapa.hasLayer(cartel)) cartel.addTo(mapa);
      });
      mapa.on('mouseout movestart zoomstart', cerrar);
      mapa.on('moveend resize', dibujar);
      dibujar();
    })();
    {% endmacro %}
    """)

    def __init__(self, pts, nivel, colores, radio, max_camiones):
        super().__init__()
        self._name = "CapaPuntos"
        self.radio = radio
        self.max_camiones = max_camiones
        fecha = pts["report_date"]
        cam_idx, camiones = pd.factorize(pts["asset_id"].astype(str))
        dia_idx, dias = pd.factorize(fecha.dt.normalize(), sort=True)
        datos = {
            "lat": np.round(pts["latitude"].to_numpy() * 1e5).astype(int).tolist(),
            "lon": np.round(pts["longitude"].to_numpy() * 1e5).astype(int).tolist(),
            "cam": cam_idx.tolist(),
            "dia": dia_idx.tolist(),
            "seg": (fecha.dt.hour * 3600 + fecha.dt.minute * 60 + fecha.dt.second).tolist(),
            "niv": nivel.tolist(),
            "camiones": list(camiones),
            "dias": [f"{DIAS[f.dayofweek]} {f:%d-%m-%Y}" for f in dias],
            "colores": colores,
        }
        self.datos = json.dumps(datos, separators=(",", ":")).replace("</", "<\\/")


# Reportes del rango de fechas: solo se quitan coordenadas invalidas y copias exactas de la ETL
# (mismo camion y misma hora). No se muestrea: van todos los reportes reales.
pts = gps_validos(df[["asset_id", "report_date", "latitude", "longitude"]].dropna())
n_validos = len(pts)
pts = pts.drop_duplicates(["asset_id", "report_date"]).sort_values(["asset_id", "report_date"])
pts = pts.assign(report_date=pd.to_datetime(pts["report_date"]))

# Concentracion de cada punto: reportes en su celda de ~CELDA_M m y las vecinas
CELDA = CELDA_M / 111_000  # metros -> grados (aprox.)
ci = np.floor(pts["latitude"].to_numpy() / CELDA).astype(np.int64)
cj = np.floor(pts["longitude"].to_numpy() / CELDA).astype(np.int64)
por_celda = pd.DataFrame({"ci": ci, "cj": cj}).value_counts().rename("n").reset_index()
alrededor = (pd.concat([por_celda.assign(ci=por_celda["ci"] + di, cj=por_celda["cj"] + dj)
                        for di in range(-RADIO_CELDAS, RADIO_CELDAS + 1)
                        for dj in range(-RADIO_CELDAS, RADIO_CELDAS + 1)])
             .groupby(["ci", "cj"])["n"].sum())
concentracion = alrededor.reindex(pd.MultiIndex.from_arrays([ci, cj])).to_numpy().astype(int)

# Tramos de color en escala logaritmica: los camiones detenidos juntan miles de reportes
# en un mismo lugar y en escala lineal todo lo demas quedaria azul.
max_conc = int(concentracion.max())
cortes = np.unique(np.geomspace(1, max_conc + 1, len(COLORES) + 1).round().astype(int))
colores = [COLORES[k] for k in np.linspace(0, len(COLORES) - 1, len(cortes) - 1).round().astype(int)]
nivel = np.clip(np.searchsorted(cortes, concentracion, side="right") - 1, 0, len(colores) - 1)

print(f"{FECHA_DESDE} a {FECHA_HASTA}: {len(df):,} filas -> {len(pts):,} puntos de "
      f"{pts['asset_id'].nunique()} camiones ({len(df) - n_validos:,} sin coordenadas validas, "
      f"{n_validos - len(pts):,} copias exactas de la ETL)")
print(f"Concentracion: mediana {np.median(concentracion):.0f} reportes alrededor · maximo {max_conc:,}")

# --- Mapa ---
mapa_puntos, _ = nuevo_mapa("Claro (Esri)", attr="Tiles &copy; Esri")

lado_m = CELDA_M * (2 * RADIO_CELDAS + 1)
folium.StepColormap(colores, index=cortes.tolist(), vmin=1, vmax=max_conc + 1,
                    caption=f"Reportes GPS en un cuadro de ~{lado_m} m (azul = pocos, rojo = muchos)"
                    ).add_to(mapa_puntos)
CapaPuntos(pts, nivel, colores, RADIO_PUNTO, MAX_CAMIONES_CARTEL).add_to(mapa_puntos)

agregar_faenas(mapa_puntos, faenas)

mapa_puntos.fit_bounds([[pts["latitude"].min(), pts["longitude"].min()],
                        [pts["latitude"].max(), pts["longitude"].max()]],
                       padding=(30, 30))
folium.LayerControl(collapsed=False).add_to(mapa_puntos)

mapa_puntos